# 01 — 核心查询循环验证

逐步验证以下模块能否协同工作：

1. `permissions/` — 规则解析 + 权限决策
2. `tools/bash.py` — BashTool 执行
3. `core/query.py` — 流式循环 + 工具调用

**运行前**：确保 `.env` 文件已配置 `ANTHROPIC_API_KEY`。

## 0. 环境初始化

In [1]:
import sys
import os
from pathlib import Path

project_root = Path(".").resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / ".env")

api_key = os.getenv("ANTHROPIC_API_KEY")
model_id = os.getenv("MODEL_ID", "claude-sonnet-4-6")

assert api_key, "未找到 ANTHROPIC_API_KEY，请在项目根目录创建 .env 文件"
print(f"API Key 已加载（前缀：{api_key[:12]}...）")
print(f"使用模型：{model_id}")

API Key 已加载（前缀：sk-0MBKxPwwk...）
使用模型：claude-sonnet-4-6


## 1. 验证权限规则解析

In [2]:
from permissions.rules import parse_rule, rule_to_string

cases = [
    "Bash",
    "Bash(git status)",
    "Bash(python -c \"print\\(1\\)\")",
    "Bash(*)",
    "Bash()",
]

for s in cases:
    result = parse_rule(s)
    roundtrip = rule_to_string(result)
    print(f"输入:     {s!r}")
    print(f"解析:     {result}")
    print(f"序列化:   {roundtrip!r}")
    print()

输入:     'Bash'
解析:     tool_name='Bash' rule_content=None
序列化:   'Bash'

输入:     'Bash(git status)'
解析:     tool_name='Bash' rule_content='git status'
序列化:   'Bash(git status)'

输入:     'Bash(python -c "print\\(1\\)")'
解析:     tool_name='Bash' rule_content='python -c "print(1)"'
序列化:   'Bash(python -c "print\\(1\\)")'

输入:     'Bash(*)'
解析:     tool_name='Bash' rule_content=None
序列化:   'Bash'

输入:     'Bash()'
解析:     tool_name='Bash' rule_content=None
序列化:   'Bash'



## 2. 验证权限决策流水线

Jupyter 内置 event loop，直接用 `await` 而非 `asyncio.run()`。

In [3]:
from permissions.manager import PermissionManager, PermissionContext
from permissions.rules import PermissionMode
from tools.bash import BashTool

bash = BashTool()

# 场景 1：bypass 模式，应直接允许
ctx = PermissionContext(mode=PermissionMode.BYPASS)
decision = await PermissionManager(ctx).check(bash, {"command": "ls"})
print(f"bypass 模式:          {decision}")

# 场景 2：deny 规则命中，应直接拒绝
ctx = PermissionContext.from_rule_strings(deny=["Bash"])
decision = await PermissionManager(ctx).check(bash, {"command": "ls"})
print(f"deny 规则命中:        {decision}")

# 场景 3：allow 规则命中，应允许
ctx = PermissionContext.from_rule_strings(mode=PermissionMode.DEFAULT, allow=["Bash"])
decision = await PermissionManager(ctx).check(bash, {"command": "ls"})
print(f"allow 规则命中:       {decision}")

# 场景 4：default 模式无规则，应返回 ask
ctx = PermissionContext(mode=PermissionMode.DEFAULT)
decision = await PermissionManager(ctx).check(bash, {"command": "rm -rf /"})
print(f"default 模式（无规则）: {decision}")

bypass 模式:          behavior='allow' reason='权限模式 bypassPermissions 允许所有工具'
deny 规则命中:        behavior='deny' reason='Bash 被拒绝规则禁止使用'
allow 规则命中:       behavior='allow' reason='allow 规则匹配：Bash'
default 模式（无规则）: behavior='ask' reason='需要用户确认是否允许 Bash 执行' rule=None


## 3. 验证 BashTool 本地执行

In [4]:
from tools.bash import BashTool
from tools.base import ToolUseContext
from permissions.manager import PermissionManager, PermissionContext
from permissions.rules import PermissionMode

bash = BashTool(timeout=10)
ctx = ToolUseContext(
    permission_manager=PermissionManager(PermissionContext(mode=PermissionMode.BYPASS)),
    model=model_id,
    tools=[bash],
    session_id="notebook-test",
)

async def run_bash(command: str):
    print(f"$ {command}")
    async for chunk in bash.execute({"command": command}, ctx):
        print(chunk, end="")
    print()

await run_bash("echo Hello from BashTool")
await run_bash("python --version")
await run_bash("ls ../")

$ echo Hello from BashTool
Hello
from
BashTool

$ python --version
Python 3.13.13

$ ls ../


    Ŀ¼: D:\github_use\claude-code-py


Mode                 LastWriteTime         Length Name                                                                 
----                 -------------         ------ ----                                                                 
d-----         2026/6/21     17:07                .venv                                                                
d-----         2026/6/21     16:58                api                                                                  
d-----         2026/6/21     20:44                core                                                                 
d-----         2026/6/21     17:45                docs                                                                 
d-----         2026/6/21     20:28                notebooks                                                            
d-----         2026/6/21   

## 4. 验证核心查询循环（无工具，纯对话）

In [5]:
from core.query import QueryParams, query, TextDeltaEvent, MessageCompleteEvent
from permissions.manager import PermissionManager, PermissionContext
from permissions.rules import PermissionMode

params = QueryParams(
    messages=[{"role": "user", "content": "用一句话介绍你自己"}],
    system_prompt="你是一个简洁的助手，回答不超过两句话。",
    model=model_id,
    api_key=api_key,
)
mgr = PermissionManager(PermissionContext(mode=PermissionMode.BYPASS))

print("模型回复：", end="", flush=True)
async for event in query(params, tools=[], permission_manager=mgr):
    if isinstance(event, TextDeltaEvent):
        print(event.text, end="", flush=True)
    elif isinstance(event, MessageCompleteEvent):
        print(f"\n\n[stop_reason={event.stop_reason}, tokens={event.usage}]")

模型回复：我是一个由Anthropic开发的AI助手Claude，致力于为您提供准确、有帮助的信息和对话支持。

[stop_reason=end_turn, tokens={'input_tokens': 39, 'output_tokens': 43, 'cache_read_input_tokens': 0}]


## 5. 验证完整工具调用循环（核心）

In [6]:
from core.query import (
    QueryParams, query,
    StreamRequestStartEvent, TextDeltaEvent,
    ToolUseEvent, ToolResultEvent, MessageCompleteEvent,
)
from tools.bash import BashTool
from permissions.manager import PermissionManager, PermissionContext
from permissions.rules import PermissionMode

bash = BashTool(timeout=15)
mgr = PermissionManager(PermissionContext(mode=PermissionMode.BYPASS))

params = QueryParams(
    messages=[{
        "role": "user",
        "content": "用 Bash 工具列出当前目录下的 .py 文件，然后告诉我有几个。"
    }],
    system_prompt="你是一个编程助手，可以使用 Bash 工具执行命令。",
    model=model_id,
    api_key=api_key,
    max_turns=5,
)

print("=== 事件流 ===")
async for event in query(params, tools=[bash], permission_manager=mgr):
    if isinstance(event, StreamRequestStartEvent):
        print("\n--- 新一轮 API 请求 ---")
    elif isinstance(event, TextDeltaEvent):
        print(event.text, end="", flush=True)
    elif isinstance(event, ToolUseEvent):
        print(f"\n[工具调用] {event.tool_name}({event.tool_input})")
    elif isinstance(event, ToolResultEvent):
        preview = event.content[:200].replace("\n", "\\n")
        print(f"[工具结果] is_error={event.is_error} | {preview}")
    elif isinstance(event, MessageCompleteEvent):
        print(f"\n[完成] stop_reason={event.stop_reason}")

=== 事件流 ===

--- 新一轮 API 请求 ---

[工具调用] Bash({'command': 'find . -maxdepth 1 -name "*.py" -type f'})

[完成] stop_reason=tool_use
\n\n[退出码: 1]\nr=False | �Ҳ����ļ� - *.py

--- 新一轮 API 请求 ---
当前目录下 **没有 `.py` 文件**（0 个）。

`find` 命令未找到任何匹配 `*.py` 的文件，说明当前目录中不存在 Python 源文件。如果你想在其他目录查找，或者递归搜索子目录，告诉我即可！ 😊
[完成] stop_reason=end_turn
